# **Setup**

* https://www.kdnuggets.com/best-small-language-models-on-hugging-face-right-now 

In [1]:
!pip install --upgrade transformers -q

In [ ]:
import os 
os.environ['HF_TOKEN'] = '<TOKEN>'

# **Alibaba Qwen 3.5-4B**

* https://huggingface.co/Qwen/Qwen3.5-4B

In [2]:
# Install: pip install transformers torch accelerate
from transformers import AutoModelForCausalLM, AutoTokenizer

# Specify the model ID from Hugging Face Hub
model_id = "Qwen/Qwen3.5-4B"

# Load the tokenizer -- handles text encoding and chat formatting
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model; torch_dtype="auto" picks the best precision
# device_map="auto" places layers across available hardware automatically
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

# Build the conversation as a list of message dicts
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain the difference between supervised and unsupervised learning in simple terms."}
]

# Apply the model's built-in chat template to format the messages correctly
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    # Setting enable_thinking=False skips the reasoning chain for faster output
    # Remove this line if you want the model to reason step by step before answering
    enable_thinking=False
)

# Tokenize and move inputs to the same device as the model
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate the response -- max_new_tokens caps output length
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

# Decode only the newly generated tokens (not the input prompt)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
response = tokenizer.decode(output_ids, skip_special_tokens=True)

print(response)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Here is the difference between supervised and unsupervised learning explained simply:

### The Core Difference: The "Answer Key"
The main difference comes down to whether the data comes with the correct answers (labels) or not.

#### 1. Supervised Learning (Learning with a Teacher)
Imagine you are teaching a child to recognize animals. You show them a picture of a **dog** and tell them, "This is a dog." Then you show a picture of a **cat** and say, "This is a cat." You provide every picture with the correct label.

*   **How it works:** The algorithm learns by looking at input data (the picture) and the corresponding output label (dog/cat). It tries to find the pattern that links the two.
*   **Goal:** To predict the label of new, unseen data.
*   **Common Analogy:** Studying for a test where you have a textbook (data) and an answer key (labels).
*   **Common Uses:** Spam filters (learning what looks like spam), house price prediction, face recognition.

#### 2. Unsupervised Learning (

# **Microsoft Phi-4-mini-instruct**

* https://huggingface.co/microsoft/Phi-4-mini-instruct

In [ ]:
# Install: pip install transformers torch

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "microsoft/Phi-4-mini-instruct"

# Load the tokenizer for Phi-4-mini
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load model in bfloat16 for memory efficiency on GPU
# Use torch_dtype=torch.float32 if running on CPU only
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)

# Phi-4-mini uses a system/user/assistant chat format
messages = [
    {"role": "system", "content": "You are a helpful assistant focused on clear, accurate answers."},
    {"role": "user", "content": "What is the difference between a list and a tuple in Python?"}
]

# Apply the model's chat template -- Phi-4-mini expects this specific formatting
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# Generate the response
outputs = model.generate(
    inputs,
    max_new_tokens=300,       # Keep responses focused
    temperature=0.7,          # Slight randomness for natural output
    do_sample=True            # Required when temperature > 0
)

# Decode and print only the generated portion
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(response)

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

AttributeError: 

# **Google Gemma E4B**

* https://huggingface.co/google/gemma-4-E4B-it 

In [3]:
# Install: pip install transformers torch

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "google/gemma-4-E4B-it"
hf_token = os.getenv("HF_TOKEN")

# Load tokenizer -- handles Gemma's specific chat format
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

# Load model; bfloat16 cuts memory roughly in half vs float32
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)

# Gemma uses a role-based chat template -- always pass messages this way
messages = [
    {"role": "user", "content": "Write a Python function that checks if a string is a palindrome."}
]

# Tokenize using the model's built-in chat template
inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True
).to(model.device)

# Run generation
with torch.no_grad():  # Disables gradient tracking -- speeds up inference
    outputs = model.generate(
        inputs,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.7
    )

# Strip the input tokens and decode just the response
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(response)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

AttributeError: 

# **Meta Llama 3.2 3B Instruct**